# Feature Engineering & Selection

Based on the EDA (`02_master_eda.ipynb`) we:

1. **Drop columns** that leak future information, are identifiers, or are redundant
2. **Aggregate inter-correlated features** into composite signals to reduce multi-collinearity
3. **Create new derived features** that capture business-meaningful ratios / changes
4. **Encode categoricals** for model readiness
5. **Export** the final modelling-ready dataset

In [26]:
import pandas as pd
import numpy as np
from pathlib import Path

project_root = Path.cwd().parent.parent
df = pd.read_csv(project_root / "data" / "processed" / "master_churn_dataset.csv", low_memory=False)
print(f"Starting shape: {df.shape}")
print(f"Target distribution:\n{df['Prospect_Outcome'].value_counts()}")

Starting shape: (122082, 138)
Target distribution:
Prospect_Outcome
Won        101226
Churned     12668
Open         8188
Name: count, dtype: int64


---
## Step 1 — Remove identifiers, date strings & leaky columns

**Why each column is dropped:**

| Column | Reason |
|:---|:---|
| `Co_Ref` | Customer identifier — not a feature |
| `Renewal_Month`, `Prospect_Renewal_Date`, `Closed_Date`, `DateTime_Out` | Date strings — not useful as raw text, temporal info already in `Renewal_Year`, `Days_To_Close_Post_Renewal` |
| `Current_Anchor_List` | Free-text list of anchors (17K unique) — too high cardinality |
| `Prospect_Status` | 46 unique values, extremely granular staging — info captured by `Prospect_Outcome` |
| `ren_competitor_benefits_mentioned` | Constant column (1 unique value = 0.0) |
| `Anchor_Group` | 100% identical to `Connection_Group` |

In [27]:
drop_cols = [
    # Identifiers & date strings
    "Co_Ref", "Renewal_Month", "Prospect_Renewal_Date", "Closed_Date", "DateTime_Out",
    # High cardinality / free text
    "Current_Anchor_List",
    # Granular status already captured by target
    "Prospect_Status",
    # Constant column (nunique=1)
    "ren_competitor_benefits_mentioned",
    # Exact duplicate of Connection_Group
    "Anchor_Group",
]

df.drop(columns=drop_cols, inplace=True)
print(f"After dropping identifiers / leaky: {df.shape}")

After dropping identifiers / leaky: (122082, 129)


---
## Step 2 — Collapse redundant price / amount columns

The EDA found **massive multi-collinearity** among 15 price columns (many at r > 0.95).  
We keep:
- `Total_Net_Paid` — the most predictive single price feature (corr = −0.48)
- `Last_Years_Price` — for computing price change

And create:
- `price_change_pct` — % change from last year's price to current total net paid
- `price_change_abs` — absolute change

In [28]:
# Create price-change features BEFORE dropping the columns
df["price_change_abs"] = df["Total_Net_Paid"] - df["Last_Years_Price"]
df["price_change_pct"] = df["price_change_abs"] / df["Last_Years_Price"].replace(0, np.nan)
df["price_change_pct"] = df["price_change_pct"].fillna(0).clip(-5, 5)  # cap extreme outliers

# Also create a net–paid ratio vs last year
df["net_paid_vs_last"] = df["Total_Net_Paid"] / df["Last_Total_Net_Paid"].replace(0, np.nan)
df["net_paid_vs_last"] = df["net_paid_vs_last"].fillna(1).clip(0, 10)

# Drop the redundant price columns
price_drop = [
    "Starting_Net", "Starting_Vat", "Starting_Gross",
    "Starting_Membership_Net", "Starting_Package_Net", "Starting_PQQ_Net",
    "Gross", "Membership_Net", "Package_Net", "PQQNet",
    "Amount", "Total_Amount",
    "Last_Years_Price", "Last_Total_Net_Paid",
]
df.drop(columns=price_drop, inplace=True)
print(f"After price collapse: {df.shape}")
print(f"New features: price_change_abs, price_change_pct, net_paid_vs_last")

After price collapse: (122082, 118)
New features: price_change_abs, price_change_pct, net_paid_vs_last


C:\Users\ShefaliChopra\AppData\Local\Temp\ipykernel_15740\1673941309.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["price_change_abs"] = df["Total_Net_Paid"] - df["Last_Years_Price"]
C:\Users\ShefaliChopra\AppData\Local\Temp\ipykernel_15740\1673941309.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["price_change_pct"] = df["price_change_abs"] / df["Last_Years_Price"].replace(0, np.nan)
C:\Users\ShefaliChopra\AppData\Local\Temp\ipykernel_15740\1673941309.py:7: PerformanceWarning: DataFrame is highly fragmented.  T

---
## Step 3 — Collapse redundant connections columns

`Proforma_Approved_Lists` and `#_of_Connection` are perfectly correlated (r = 1.0).  
`Current_Anchorings` is at r = 0.92 with them.  
We keep `#_of_Connection` and create a `connection_change` feature.

In [4]:
# Connection change from last year
df["connection_change"] = df["#_of_Connection"] - df["Last_Connections"]

# Drop redundant
df.drop(columns=["Current_Anchorings", "Proforma_Approved_Lists", "Last_Connections"], inplace=True)
print(f"After connection collapse: {df.shape}")

After connection collapse: (122082, 116)


---
## Step 4 — Aggregate inter-correlated email CRM flags

The EDA found strong clusters among the 19 email CRM binary-proportion features.  
We group them into **4 composite signals** (mean of their component flags):

| Composite | Components | Rationale |
|:---|:---|:---|
| `em_accreditation_health` | accreditation_completed, progress, delays, contractor_engagement, dts_or_ssip, accreditation_issues | Accreditation journey health |
| `em_churn_risk_signals` | contractor_suggested_leave, competitors_mentioned, refund_mentioned, financial_hardship | Direct churn-intent signals |
| `em_dissatisfaction_index` | customer_complained, negative_experience, dissatisfaction_with_support, dissatisfied_with_renewal_price, platform_issues_raised | Unhappiness composite |
| `em_engagement_signals` | timely_completion, customer_payment_intention, agent_chased_contractor, membership_overdue | Engagement / responsiveness |

We also keep the **standalone top predictor** `em_crm_contractor_suggested_leave` as-is (corr = 0.26), because it's so powerful.

In [5]:
# Accreditation health composite
accred_cols = [
    "em_crm_accreditation_completed", "em_crm_progress_towards_accreditation",
    "em_crm_delays_in_accreditation", "em_crm_contractor_engagement",
    "em_crm_dts_or_ssip_mentioned", "em_crm_accreditation_issues"
]
df["em_accreditation_health"] = df[accred_cols].mean(axis=1)

# Churn risk signals composite
churn_risk_cols = [
    "em_crm_contractor_suggested_leave", "em_crm_competitors_mentioned",
    "em_crm_refund_mentioned", "em_crm_financial_hardship_mentioned"
]
df["em_churn_risk_signals"] = df[churn_risk_cols].mean(axis=1)

# Dissatisfaction index composite
dissat_cols = [
    "em_crm_customer_complained", "em_crm_negative_customer_experience",
    "em_crm_dissatisfaction_with_support", "em_crm_dissatisified_with_renewal_price",
    "em_crm_platform_issues_raised"
]
df["em_dissatisfaction_index"] = df[dissat_cols].mean(axis=1)

# Engagement signals composite
engage_cols = [
    "em_crm_timely_completion", "em_crm_customer_payment_intention",
    "em_crm_agent_chased_contractor", "em_crm_membership_overdue"
]
df["em_engagement_signals"] = df[engage_cols].mean(axis=1)

# Keep the single strongest CRM predictor as a standalone
# em_crm_contractor_suggested_leave is already kept

# Drop the individual components (except contractor_suggested_leave)
individual_em_drop = list(set(
    accred_cols + churn_risk_cols + dissat_cols + engage_cols
) - {"em_crm_contractor_suggested_leave"})

df.drop(columns=individual_em_drop, inplace=True)
print(f"After email CRM aggregation: {df.shape}")
print(f"New composites: em_accreditation_health, em_churn_risk_signals, em_dissatisfaction_index, em_engagement_signals")

After email CRM aggregation: (122082, 102)
New composites: em_accreditation_health, em_churn_risk_signals, em_dissatisfaction_index, em_engagement_signals


---
## Step 5 — Aggregate inter-correlated CC call flags

CC features had the weakest correlations with churn overall (none in top 30).  
We aggregate the 20 individual flags into **3 composites** and keep the sentiment scores.

In [6]:
# CC Dissatisfaction composite
cc_dissat = [
    "cc_customer_issues_concerns", "cc_business_struggles_financial_hardship",
    "cc_dissatisfaction_time_to_complete", "cc_process_complexity_concerns",
    "cc_questions_harder_than_expected", "cc_dissatisfaction_support",
    "cc_contractor_complained", "cc_contractor_suggest_leave"
]
df["cc_dissatisfaction_index"] = df[cc_dissat].mean(axis=1)

# CC Platform / technical issues composite
cc_tech = [
    "cc_issues_within_questionnaire", "cc_login_issues", "cc_platform_issues"
]
df["cc_platform_issues_index"] = df[cc_tech].mean(axis=1)

# CC Pricing / commercial composite
cc_price = [
    "cc_pricing_mentioned", "cc_refund_discussed", "cc_pricing_sentiment_impact"
]
df["cc_pricing_index"] = df[cc_price].mean(axis=1)

# CC Engagement composite
cc_engage = [
    "cc_care_package_discussed", "cc_urgency_getting_on_site",
    "cc_external_consultant", "cc_agent_cross_sell_attempt",
    "cc_questionnaire_completion", "cc_chasing_response"
]
df["cc_engagement_index"] = df[cc_engage].mean(axis=1)

# Collapse CC sentiment scores to a single composite
# cc_sent_start, end, overall, issues are inter-correlated
cc_sent_cols = ["cc_sent_start_score_mean", "cc_sent_end_score_mean",
                "cc_sent_overall_score_mean", "cc_sent_issues_score_mean"]
df["cc_sentiment_score_avg"] = df[cc_sent_cols].mean(axis=1)

# Drop individual CC flag columns and individual sentiment scores
cc_drop = cc_dissat + cc_tech + cc_price + cc_engage + cc_sent_cols
df.drop(columns=cc_drop, inplace=True)
print(f"After CC aggregation: {df.shape}")
print(f"New composites: cc_dissatisfaction_index, cc_platform_issues_index, cc_pricing_index, cc_engagement_index, cc_sentiment_score_avg")

After CC aggregation: (122082, 83)
New composites: cc_dissatisfaction_index, cc_platform_issues_index, cc_pricing_index, cc_engagement_index, cc_sentiment_score_avg


---
## Step 6 — Aggregate inter-correlated renewal call features

Key redundancies found:
- `ren_other_complaint` ↔ `ren_has_complaint` (r = 0.98) — keep composite
- `ren_friction_score_mean` ↔ `ren_friction_score_max` (r = 0.99) — keep mean only
- `ren_price_switching_mentioned` ↔ `ren_competitor_value_comparison` (r = 0.79) — combine
- Multiple complaint/reaction features overlap heavily

In [7]:
# Renewal complaint composite (other_complaint ≈ has_complaint, serious_complaint is rare but important)
df["ren_complaint_index"] = df[["ren_other_complaint", "ren_serious_complaint", "ren_has_complaint"]].mean(axis=1)

# Renewal price-sensitivity composite
ren_price_cols = [
    "ren_discussion_on_price_increase", "ren_renewal_impact_due_to_price_increase",
    "ren_discount_or_waiver_requested", "ren_discount_offered",
    "ren_customer_asked_for_justification"
]
df["ren_price_sensitivity"] = df[ren_price_cols].mean(axis=1)

# Renewal competitor-threat composite
ren_comp_cols = [
    "ren_explicit_competitor_mention", "ren_explicit_switching_intent",
    "ren_price_switching_mentioned", "ren_competitor_value_comparison"
]
df["ren_competitor_threat"] = df[ren_comp_cols].mean(axis=1)

# Keep friction_score_mean, drop friction_score_max (r=0.99)
# Keep call_length_mean (correlated with friction but adds info)
# Keep ren_has_negative_reaction (standalone signal)

# Drop components
ren_drop = (
    ["ren_other_complaint", "ren_serious_complaint", "ren_has_complaint"]
    + ren_price_cols
    + ren_comp_cols
    + ["ren_friction_score_max"]  # near-duplicate of friction_score_mean
)
df.drop(columns=ren_drop, inplace=True)
print(f"After renewal aggregation: {df.shape}")
print(f"New composites: ren_complaint_index, ren_price_sensitivity, ren_competitor_threat")

After renewal aggregation: (122082, 73)
New composites: ren_complaint_index, ren_price_sensitivity, ren_competitor_threat


---
## Step 7 — Collapse highly-correlated billing scores

`Total_Renewal_Score_New` and `Status_Scores` are at r = 0.96 (one is nearly a sub-component of the other).  
We keep `Total_Renewal_Score_New` (the composite) and drop `Status_Scores`.

Similarly, `Renewal_Score_At_Release` has moderate correlation with `Anchoring_Score` (r = 0.59) and `Tenure_Scores` (r = 0.60).  
We keep all three since each captures a distinct signal and the correlations are moderate.

Also drop `Last_Band` — redundant with `Band` (same info from prior year, high overlap).

In [ ]:
df.drop(columns=["Status_Scores", "Last_Band"], inplace=True)
print(f"After score collapse: {df.shape}")

After score collapse: (122082, 71)


---
## Step 8 — Collapse high-cardinality renewal categorical modes

Many `ren_*_mode` categorical columns have 15-28 unique values with most rows = "No Interaction" or "Not Mentioned".  
We simplify these by keeping only the most discriminative ones and collapsing the granular ones.

In [9]:
# These high-cardinality mode columns add minimal signal vs complexity.
# Keep only the most impactful ones:
# - ren_membership_renewal_decision_mode (4 values, very discriminative)
# - ren_churn_category_mode (simplify to binary: has_churn_reason)
# - ren_customer_response_mode (3 values, clean)

# Simplify ren_churn_category_mode → binary: was a churn reason mentioned?
df["ren_has_churn_reason"] = (~df["ren_churn_category_mode"].isin(["No Interaction", "Not Mentioned"])).astype(int)

# Drop high-cardinality renewal mode columns
ren_cat_drop = [
    "ren_churn_category_mode",
    "ren_complaint_category_mode",
    "ren_customer_reaction_category_mode",
    "ren_agent_renewal_pitch_category_mode",
    "ren_customer_renewal_response_category_mode",
    "ren_agent_response_category_mode",
    "ren_justification_category_mode",
    "ren_reason_for_renewal_category_mode",
    "ren_agent_response_to_cancel_category_mode",
    "ren_argument_that_convinced_customer_to_stay_category_mode",
]
df.drop(columns=ren_cat_drop, inplace=True)
print(f"After renewal categorical cleanup: {df.shape}")

After renewal categorical cleanup: (122082, 62)


---
## Step 9 — Derive additional business features

In [10]:
# 1. Auto-renewal risk: combines billing auto-renewal score with token presence
df["has_auto_renewal"] = ((df["Proforma_Auto_Renewal"] == "Yes") &
                          (df["Current_Auto_Renewal_Flag"] == "Yes")).astype(int)

# 2. Has WorldPay token — payment readiness
df["has_worldpay_token"] = ((df["Proforma_World_Pay_Token"] == "Yes") |
                            (df["Current_World_Pay_Token"] == "Yes")).astype(int)

# Drop the redundant individual auto-renewal / token columns
df.drop(columns=["Proforma_Auto_Renewal", "Proforma_World_Pay_Token",
                 "Current_Auto_Renewal_Flag", "Current_World_Pay_Token"], inplace=True)

# 3. Total interaction count across all channels
df["total_interaction_count"] = df["em_email_count"] + df["cc_call_count"] + df["ren_call_count"]

# 4. Fill Days_To_Close_Post_Renewal nulls (Open prospects) with -1 sentinel
df["Days_To_Close_Post_Renewal"] = df["Days_To_Close_Post_Renewal"].fillna(-1)

print(f"After derived features: {df.shape}")
print(f"New: has_auto_renewal, has_worldpay_token, total_interaction_count")

After derived features: (122082, 61)
New: has_auto_renewal, has_worldpay_token, total_interaction_count


---
## Step 9 — Final feature inventory & export

In [11]:
# Separate target
print(f"Final shape: {df.shape}")
print(f"\nTarget: Prospect_Outcome")
print(f"\n--- Numeric features ({df.select_dtypes(include=[np.number]).shape[1]}) ---")
for col in sorted(df.select_dtypes(include=[np.number]).columns):
    print(f"  {col}")

print(f"\n--- Categorical features ---")
cat_cols = [c for c in df.columns if df[c].dtype.name in ['str', 'object'] and c != 'Prospect_Outcome']
for col in sorted(cat_cols):
    print(f"  {col} ({df[col].nunique()} unique)")

Final shape: (122082, 118)

Target: Prospect_Outcome

--- Numeric features (90) ---
  #_of_Connection
  Anchoring_Score
  Auto_Renewal_Score
  Current_Anchorings
  Days_To_Close_Post_Renewal
  Last_Connections
  Payment_Timeframe
  Proforma_Approved_Lists
  Renewal_Score_At_Release
  Renewal_Year
  Status_Scores
  Sustainability_Score
  Tenure_Scores
  Tenure_Years
  Total_Net_Paid
  Total_Renewal_Score_New
  cc_agent_cross_sell_attempt
  cc_business_struggles_financial_hardship
  cc_call_count
  cc_care_package_discussed
  cc_chasing_response
  cc_contractor_complained
  cc_contractor_suggest_leave
  cc_customer_issues_concerns
  cc_dissatisfaction_support
  cc_dissatisfaction_time_to_complete
  cc_external_consultant
  cc_issues_within_questionnaire
  cc_login_issues
  cc_platform_issues
  cc_pricing_mentioned
  cc_pricing_sentiment_impact
  cc_process_complexity_concerns
  cc_questionnaire_completion
  cc_questions_harder_than_expected
  cc_refund_discussed
  cc_sent_end_score_mean


In [15]:
# Verify: correlation of new composites with churn
df_check = df[df["Prospect_Outcome"].isin(["Won", "Churned"])].copy()
df_check["churn_flag"] = (df_check["Prospect_Outcome"] == "Churned").astype(int)

num_cols = df_check.select_dtypes(include=[np.number]).columns.drop("churn_flag", errors="ignore")
corrs = df_check[num_cols].corrwith(df_check["churn_flag"]).abs().sort_values(ascending=False)

print("Top 30 features by |correlation| with churn:")
print(corrs.head(30).to_string())

Top 30 features by |correlation| with churn:
Total_Renewal_Score_New              0.673765
price_change_abs                     0.585395
net_paid_vs_last                     0.580141
Total_Net_Paid                       0.483233
price_change_pct                     0.444888
Tenure_Scores                        0.377107
Renewal_Score_At_Release             0.315323
Sustainability_Score                 0.302563
em_crm_contractor_suggested_leave    0.262268
Auto_Renewal_Score                   0.240549
Days_To_Close_Post_Renewal           0.220009
Anchoring_Score                      0.215078
em_churn_risk_signals                0.214802
ren_call_count                       0.200889
ren_max_call_number                  0.182283
ren_has_churn_reason                 0.174748
Tenure_Years                         0.162347
ren_friction_score_mean              0.157236
has_worldpay_token                   0.156730
em_auto_renewal_status_max           0.155947
ren_call_length_mean               

In [13]:
import os

out_dir = project_root / "data" / "processed"
out_path = out_dir / "model_ready_dataset.csv"
df.to_csv(out_path, index=False)
print(f"Saved to: {out_path}")
print(f"Final shape: {df.shape}")
print(f"File size: {os.path.getsize(out_path) / 1e6:.1f} MB")

Saved to: c:\Users\ShefaliChopra\Churn-Prediction-JMAN\data\processed\model_ready_dataset.csv
Final shape: (122082, 61)
File size: 41.7 MB


---
## Summary of Changes

| Action | Before | After | Details |
|:---|:---:|:---:|:---|
| Drop identifiers & leaky cols | 138 | 129 | Co_Ref, dates, Anchor_Group, etc. |
| Collapse price columns | 129 | 118 | 15 price cols → `Total_Net_Paid` + 3 derived |
| Collapse connection cols | 118 | 116 | 4 conn cols → `#_of_Connection` + `connection_change` |
| Aggregate email CRM flags | 116 | 101 | 19 flags → 4 composites + 1 standalone |
| Aggregate CC call flags | 101 | 80 | 24 flags → 5 composites |
| Aggregate renewal flags | 80 | 68 | 12 flags → 3 composites |
| Collapse billing scores | 68 | 66 | Drop Status_Scores, Last_Band |
| Collapse renewal categoricals | 66 | 57 | 10 mode cols → 2 kept + 1 binary |
| Derive business features | 57 | ~55 | auto_renewal, token, interaction count |

**Result:** ~138 → ~55 features with reduced multi-collinearity and business-meaningful composites.